# 01 — Echo smoke

End-to-end check that the egras stack works for a realistic B2B integration:

1. Operator logs in.
2. Operator provisions a new tenant org.
3. Operator switches into the tenant org and creates a service account.
4. Operator grants the service account `org_admin` (which carries `echo:invoke` after migration 0014).
5. Operator mints an API key restricted to `scopes=['echo:invoke']`.
6. Service-account caller hits `POST /api/v1/echo` and asserts the payload round-trips with the right org and key id.

Two negative checks follow:

7. A second key without `echo:invoke` is rejected with 403.
8. The operator overrides the org's `auth.api_key_headers` flag to `['x-api-key']` only; the same key sent via `Authorization: Bearer` returns 401, while `X-API-Key` still works.

In [ ]:
import os, sys, time, requests

# Allow `from lib.egras import …` whether the notebook is run from the repo root
# (pytest --nbmake) or from notebooks/scenarios/ (jupyter notebook).
for candidate in (os.path.abspath('..'), os.path.abspath('notebooks')):
    if candidate not in sys.path:
        sys.path.insert(0, candidate)

from lib.egras import (
    Client,
    add_user_to_org,
    create_org,
    create_service_account,
    login_operator,
    mint_api_key,
    operator_credentials,
)

BASE = os.environ.get('EGRAS_BASE_URL', 'http://localhost:8080')

## Operator login + org provisioning

In [ ]:
email, password = operator_credentials()
op_jwt = login_operator(BASE, email, password)
op_home = Client(BASE, jwt=op_jwt)  # stays in operator org — has tenants.manage_all

org = create_org(op_home, name=f'AcmeCorp-{int(time.time())}')
org_id = org['id']
print('created org', org_id)

## Switch into the tenant org and create a service account

In [ ]:
switch = op_home.post('/api/v1/security/switch-org', json={'org_id': org_id})
switch.raise_for_status()
op_target = Client(BASE, jwt=switch.json()['token'])

sa = create_service_account(op_target, org_id, 'echo-bot')
sa_user_id = sa['user_id']
add_user_to_org(op_target, org_id, sa_user_id, 'org_admin')

plaintext, key_meta = mint_api_key(op_target, sa_user_id, scopes=['echo:invoke'])
print('minted key', key_meta['key']['id'])

## Positive case — payload round-trips through `/api/v1/echo`

In [ ]:
caller = Client(BASE, api_key=plaintext)
resp = caller.post('/api/v1/echo', json={'hello': 'world'})

assert resp.status_code == 200, resp.text
body = resp.json()
assert body['method'] == 'POST'
assert body['payload'] == {'hello': 'world'}
assert body['org_id'] == org_id
assert body['key_id'] == key_meta['key']['id']
assert body['principal_user_id'] == sa_user_id
print('echo round-trip OK', body)

## Negative case — a key without `echo:invoke` is rejected with 403

In [ ]:
plaintext_no_perm, _ = mint_api_key(
    op_target, sa_user_id, name='no-echo', scopes=['features.read']
)
denied = Client(BASE, api_key=plaintext_no_perm).post(
    '/api/v1/echo', json={'x': 1}
)
assert denied.status_code == 403, denied.text
assert denied.json()['type'].endswith('/permission.denied')
print('rejected without echo:invoke OK')

## Header allowlist demo — restrict the org to `X-API-Key` only

The flag `auth.api_key_headers` is not self-service, so the operator (with `tenants.manage_all`) sets the override from their home-org JWT (`op_home`). Once set, the same API key sent via `Authorization: Bearer …` is rejected with 401, while `X-API-Key: …` continues to work.

In [ ]:
put = op_home.put(
    f'/api/v1/features/orgs/{org_id}/auth.api_key_headers',
    json={'value': ['x-api-key']},
)
assert put.status_code == 200, put.text
assert put.json()['value'] == ['x-api-key']

via_bearer = requests.post(
    f'{BASE}/api/v1/echo',
    headers={'authorization': f'Bearer {plaintext}', 'content-type': 'application/json'},
    json={'should': 'fail'},
)
assert via_bearer.status_code == 401, via_bearer.text

via_x_api_key = caller.post('/api/v1/echo', json={'still': 'works'})
assert via_x_api_key.status_code == 200, via_x_api_key.text
print('header allowlist enforced — bearer 401, x-api-key 200')